# Gold Layer - Dimensional Model

## Schema
- DimDate: shared across both facts
- DimRegion: regional unemployment breakdown
- FactUnemployment: monthly unemployment rate by region
- FactMacroIndicators: monthly CPI + SNB policy rate (national)

## Design decisions
- Two fact tables: different geographic grain (regional vs national)
- Linked via DimDate in Power BI for cross-indicator analysis
- Surrogate keys on all dimensions

In [1]:
import pandas as pd
from pathlib import Path

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold")
GOLD_PATH.mkdir(parents=True, exist_ok=True)

# Load silver
df_unemployment = pd.read_parquet(SILVER_PATH / "silver_unemployment.parquet")
df_cpi = pd.read_parquet(SILVER_PATH / "silver_cpi.parquet")
df_policy = pd.read_parquet(SILVER_PATH / "silver_policy_rate.parquet")

print(f"Silver unemployment : {df_unemployment.shape}")
print(f"Silver CPI          : {df_cpi.shape}")
print(f"Silver policy rate  : {df_policy.shape}")

Silver unemployment : (1365, 9)
Silver CPI          : (394, 7)
Silver policy rate  : (197, 7)


In [2]:
# ── DimDate ──────────────────────────────────────────────────────────────────

def build_dim_date(periods: pd.Series) -> pd.DataFrame:
    """
    Build DimDate from a series of YYYY-MM period strings.
    Surrogate key: integer YYYYMM (e.g. 202301)
    """
    unique_periods = periods.drop_duplicates().sort_values().reset_index(drop=True)
    
    dim_date = pd.DataFrame({"period": unique_periods})
    dim_date["date_id"]  = dim_date["period"].str.replace("-", "").astype(int)
    dim_date["year"]     = dim_date["period"].str[:4].astype(int)
    dim_date["month"]    = dim_date["period"].str[5:7].astype(int)
    dim_date["quarter"]  = ((dim_date["month"] - 1) // 3 + 1).astype(int)
    dim_date["year_month_label"] = dim_date["period"]  # for Power BI axis labels
    
    return dim_date[["date_id", "period", "year", "month", "quarter", "year_month_label"]]


# Combine all periods from all sources
all_periods = pd.concat([
    df_unemployment["period"],
    df_cpi["period"],
    df_policy["period"]
])

dim_date = build_dim_date(all_periods)

print(f"DimDate : {len(dim_date)} rows")
print(f"\nSample:\n{dim_date.head(4)}")
print(f"\nRange: {dim_date['period'].min()} → {dim_date['period'].max()}")

DimDate : 197 rows

Sample:
   date_id   period  year  month  quarter year_month_label
0   201001  2010-01  2010      1        1          2010-01
1   201002  2010-02  2010      2        1          2010-02
2   201003  2010-03  2010      3        1          2010-03
3   201004  2010-04  2010      4        2          2010-04

Range: 2010-01 → 2026-05
